# PhishShield AI — Phishing URL Detection with Machine Learning

**IBM SkillsBuild Cyber Security 2026 | BharatCares — Final Project**  
**Submitted by:** Brinda Kachchi  
**Project type:** Defensive cybersecurity / machine learning

This notebook builds and evaluates a phishing-URL risk classifier using the **PhiUSIIL Phishing URL (Website)** dataset from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset).

**Important:** This model is an educational screening tool, **not** a website safety guarantee. No website is visited or clicked during feature extraction or prediction.

**How to run:** Install `requirements.txt`, start Jupyter Notebook or JupyterLab, and execute **Run All** in order. The first data-loading run requires an internet connection.

## 1. Objectives

1. Collect labeled legitimate and phishing URLs from a cited public dataset.
2. Extract structural and lexical URL features without contacting websites.
3. Reduce duplicate-URL contamination and hold out unseen hostnames for evaluation.
4. Compare Logistic Regression and Random Forest using phishing-class precision, recall, F1, and ROC-AUC.
5. Inspect feature importance, test example strings safely, and save a reusable model.

In [ ]:
# Core imports and reproducibility
from collections import Counter
from math import log2
from urllib.parse import urlsplit, parse_qsl
from pathlib import Path
import ipaddress
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
import joblib

SEED = 42
MAX_SAMPLES_PER_CLASS = 10000  # 20,000 total max; set to None for all data
MODEL_PATH = Path("phishshield_model.joblib")
print("Environment ready; random seed:", SEED)

## 2. Load and audit the public dataset

**Source:** Prasad & Chandra, *PhiUSIIL Phishing URL (Website)*, UCI ML Repository (ID 967). Original labels are **0 = phishing**, **1 = legitimate**. For our classifier we remap to **1 = phishing**, **0 = legitimate** to make phishing the positive class.

The notebook downloads labels and raw URL strings. It deliberately does **not** use HTML/page-derived UCI features that cannot be reproduced from a URL entered by a user.

In [ ]:
dataset = fetch_ucirepo(id=967)
raw_features = dataset.data.features
raw_targets = dataset.data.targets

if raw_features is None or "URL" not in raw_features.columns:
    raise ValueError("Expected a URL column in UCI PhiUSIIL feature data.")
if raw_targets is None or raw_targets.shape[1] == 0:
    raise ValueError("Expected a target column in UCI PhiUSIIL target data.")

records = pd.DataFrame({
    "url": raw_features["URL"].astype("string").str.strip(),
    "original_label": pd.to_numeric(raw_targets.iloc[:, 0], errors="coerce"),
})
records = records.dropna(subset=["url", "original_label"]).copy()
records = records[records["original_label"].isin([0, 1])]
records = records[records["url"].str.len() > 0]

# Ambiguous duplicate URLs (same URL with conflicting labels) are removed.
conflicts = records.groupby("url")["original_label"].nunique()
ambiguous_urls = set(conflicts[conflicts > 1].index)
records = records[~records["url"].isin(ambiguous_urls)]
records = records.drop_duplicates(subset=["url"]).reset_index(drop=True)
records["is_phishing"] = (records["original_label"] == 0).astype(int)

print("Clean unique URL records:", len(records))
print("Conflicting URLs removed:", len(ambiguous_urls))
print("Class counts (0=legitimate, 1=phishing):", records["is_phishing"].value_counts().to_dict())
records[["is_phishing"]].value_counts().rename("count").to_frame()

## 3. Safe URL feature engineering

All features come from the text of the URL, **without HTTP requests or DNS lookups**. Suspicious words, IP-address hostnames, extra delimiters, and entropy are *signals*, not proof of fraud. The model must learn their joint statistical associations from real labels.

In [ ]:
SUSPICIOUS_TOKENS = (
    "login", "verify", "account", "secure", "update", "password",
    "signin", "confirm", "reset", "billing", "wallet", "payment"
)

def safe_split(url):
    """Parse an input URL without opening it; tolerate missing scheme."""
    value = str(url).strip()
    if not re.match(r"^[A-Za-z][A-Za-z0-9+.-]*://", value):
        value = "http://" + value
    try:
        return urlsplit(value)
    except ValueError:
        return urlsplit("http://invalid.example.invalid/")

def get_hostname(url):
    try:
        return (safe_split(url).hostname or "").lower()
    except ValueError:
        return ""

def shannon_entropy(value):
    if not value:
        return 0.0
    counts = Counter(value)
    total = len(value)
    return -sum((count / total) * log2(count / total) for count in counts.values())

def extract_url_features(url):
    url = str(url).strip()
    parts = safe_split(url)
    host = get_hostname(url)
    path = parts.path or ""
    query = parts.query or ""
    text = url.lower()
    digits = sum(char.isdigit() for char in url)
    host_labels = [piece for piece in host.split(".") if piece]
    try:
        ipaddress.ip_address(host)
        ip_host = 1
    except ValueError:
        ip_host = 0
    try:
        has_port = int(parts.port is not None)
    except ValueError:
        has_port = 0
    return {
        "url_length": len(url),
        "hostname_length": len(host),
        "path_length": len(path),
        "query_length": len(query),
        "digit_count": digits,
        "digit_ratio": digits / max(len(url), 1),
        "dot_count": url.count("."),
        "hyphen_count": url.count("-"),
        "underscore_count": url.count("_"),
        "at_count": url.count("@"),
        "percent_count": url.count("%"),
        "slash_count": url.count("/"),
        "equals_count": url.count("="),
        "question_count": url.count("?"),
        "subdomain_count": max(0, len(host_labels) - 2),
        "query_parameter_count": len(parse_qsl(query, keep_blank_values=True)),
        "uses_https": int(parts.scheme.lower() == "https"),
        "ip_hostname": ip_host,
        "has_port": has_port,
        "has_punycode": int("xn--" in host),
        "suspicious_token_count": sum(token in text for token in SUSPICIOUS_TOKENS),
        "hostname_entropy": shannon_entropy(host),
    }

# Checks exercise the extractor, including a harmless, reserved test domain.
test_input = "http://secure-login.example.invalid/verify?user=123"
feature_check = extract_url_features(test_input)
assert feature_check["uses_https"] == 0
assert feature_check["query_parameter_count"] == 1
assert feature_check["suspicious_token_count"] >= 2
assert all(np.isfinite(list(feature_check.values())))
print("Feature extraction self-check passed;", len(feature_check), "features.")

## 4. Sampling and hostname-separated holdout

To keep laptop runtime manageable, the notebook reproducibly samples at most 10,000 URLs per class. **Hostnames are grouped into either train or test, never both**, helping reduce memorization of exact domains. This is stricter than a naive random URL split, though related domains and time drift may still cause optimistic performance.

In [ ]:
valid = records.copy()
valid["hostname"] = valid["url"].map(get_hostname)
valid = valid[valid["hostname"].str.len() > 0].copy()

if MAX_SAMPLES_PER_CLASS is not None:
    valid = pd.concat([
        group.sample(n=min(len(group), MAX_SAMPLES_PER_CLASS), random_state=SEED)
        for _, group in valid.groupby("is_phishing")
    ], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

X = pd.DataFrame([extract_url_features(url) for url in valid["url"]])
y = valid["is_phishing"].astype(int).to_numpy()
groups = valid["hostname"].to_numpy()
feature_names = X.columns.tolist()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

assert len(set(groups[train_idx]) & set(groups[test_idx])) == 0
assert len(set(y_train)) == 2 and len(set(y_test)) == 2, "Both classes required in each split."
print("Training rows:", len(X_train), "Test rows:", len(X_test))
print("Train counts:", Counter(y_train), "Test counts:", Counter(y_test))
print("Shared hostnames:", len(set(groups[train_idx]) & set(groups[test_idx])))

## 5. Train and compare two classifiers

Logistic Regression supplies a linear baseline; Random Forest can capture nonlinear interactions. We use **phishing-class F1** as the selection metric. The held-out test set is used for a single educational comparison, not hyperparameter tuning.

In [ ]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=120, min_samples_leaf=2, class_weight="balanced_subsample",
        random_state=SEED, n_jobs=-1
    ),
}

model_predictions = {}
summary_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    model_predictions[name] = predicted
    summary_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predicted),
        "phishing_precision": precision_score(y_test, predicted, zero_division=0),
        "phishing_recall": recall_score(y_test, predicted, zero_division=0),
        "phishing_f1": f1_score(y_test, predicted, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
    })

results = pd.DataFrame(summary_rows).sort_values("phishing_f1", ascending=False)
display(results.style.format({col: "{:.3f}" for col in results.columns if col != "model"}))
best_name = results.iloc[0]["model"]
best_model = models[best_name]
print("Selected by phishing-class F1:", best_name)

## 6. Evaluation: confusion matrix and importance

The printed results are computed from your actual dataset run; this notebook does **not** insert predetermined accuracy values. False negatives (phishing classified as legitimate) require special caution.

In [ ]:
print(classification_report(y_test, model_predictions[best_name], target_names=["Legitimate", "Phishing"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, model_predictions[best_name],
    labels=[0, 1], display_labels=["Legitimate", "Phishing"], cmap="Blues", values_format="d"
)
plt.title("Held-out hostname-group test set")
plt.tight_layout()
plt.show()

forest = models["Random Forest"]
importances = pd.Series(forest.feature_importances_, index=feature_names).sort_values(ascending=False).head(12)
ax = importances.sort_values().plot(kind="barh", figsize=(8, 5), color="#2676b8")
ax.set_title("Random Forest: top structural URL feature importances")
ax.set_xlabel("Impurity-based importance (not causal explanation)")
plt.tight_layout()
plt.show()

## 7. Safe local examples and model export

The domains below are reserved demonstration strings, not links to visit. Risk scores are **model probabilities**, not calibrated guarantees or evidence that a site is malicious. Always verify a link using independent trusted channels.

In [ ]:
def predict_url(url, model=best_model):
    """Classify the URL string only. Makes NO network requests."""
    row = pd.DataFrame([extract_url_features(url)])[feature_names]
    probability = float(model.predict_proba(row)[0, 1])
    return {
        "url": url,
        "predicted_label": "phishing" if probability >= 0.5 else "legitimate",
        "estimated_phishing_probability": round(probability, 3),
    }

sample_strings = [
    "https://www.example.org/about",
    "http://secure-account.example.invalid/verify?session=001",
    "https://portal.example.net/login",
]
example_results = pd.DataFrame([predict_url(u) for u in sample_strings])
display(example_results)

joblib.dump({
    "model": best_model,
    "feature_names": feature_names,
    "positive_class": "phishing",
    "dataset": "UCI PhiUSIIL (id=967)",
    "seed": SEED,
}, MODEL_PATH)
print("Saved model to:", MODEL_PATH.resolve())

## 8. Conclusions and limitations

- This project demonstrates a reproducible **defensive** phishing URL triage pipeline.
- The UCI source gives real labels; collected URL strings are parsed locally without connecting to the target sites.
- Hostname-group splitting reduces but does not eliminate leakage and data-shift concerns.
- Some legitimate sites naturally contain the same tokens and patterns as phishing sites; false positives and false negatives are unavoidable.
- External validation, time-based splits, threshold calibration, and privacy/security review are needed before any production use.

**Reference:** Prasad, A., & Chandra, S. (2024), *PhiUSIIL Phishing URL (Website)*, UCI Machine Learning Repository, [dataset page](https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset). Dataset licensed CC BY 4.0. [Original paper DOI](https://doi.org/10.1016/j.cose.2023.103545).